[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_9/Übungen/01_tag_9_uebung_eigene_objekterkennung.ipynb)

# Übung: Eigener Datensatz für Objekterkennung

Sie erstellen einen kleinen, eigenen Object-Detection-Datensatz, versehen jedes Bild mit einer Bounding Box und trainieren darauf einen Detektor. Entscheidend ist nicht nur eine gute Trainingskurve: Das finale Modell muss auf **vorher nicht verwendeten Testbildern** geprüft werden.

Diese Übung passt zum transparenten Ein-Objekt-Detektor aus `01_tag_9_objekterkennung_einfache_anker_nms.ipynb`. Deshalb enthält jedes Bild genau **ein** Objekt der Zielklasse.

## Lernziele

- einen Bilddatensatz mit einem klaren Aufnahmeprotokoll erzeugen,
- Bounding Boxes im Format `[x_min, y_min, x_max, y_max]` erstellen und prüfen,
- Training, Validierung und Test sauber trennen,
- einen Detektor mit Transfer Learning trainieren und anhand von IoU bewerten,
- Fehler auf ungesehenen Bildern analysieren und gezielt verbessern.

**Mindestumfang:** 90 Bilder, davon mindestens 60 Training, 15 Validierung und 15 Test. Mehr Variation ist meist wertvoller als viele fast identische Bilder.

## Aufgabe 1 – Szenario und Datenerzeugung planen

Wählen Sie genau eine Zielklasse und ein Objekt, zum Beispiel `Tasse`, `Stift`, `Schere`, `Fernbedienung` oder `Hand`. Das Objekt soll in jedem Bild genau einmal vorkommen.

Wählen Sie einen der beiden Wege:

1. **Kamera/Smartphone:** Fotografieren Sie das Objekt. Variieren Sie Abstand, Blickwinkel, Position im Bild, Hintergrund, Licht und teilweise Verdeckung.
2. **Generierte Bilder:** Erzeugen Sie Bilder selbst, etwa mit PIL, einem Bildgenerator oder ausgeschnittenen Objekten auf wechselnden Hintergründen. Dokumentieren Sie die Erzeugungsregel. Bei programmgenerierten Bildern können die Box-Koordinaten direkt mitgespeichert werden; kontrollieren Sie trotzdem jedes Label visuell.

Nehmen Sie Testbilder bewusst getrennt auf bzw. erzeugen Sie sie mit neuen Hintergründen, Blickwinkeln oder Lichtverhältnissen. Einzelne aufeinanderfolgende Frames eines Videos dürfen nicht auf unterschiedliche Splits verteilt werden.

**Auszufüllen:**

- Zielklasse / Objekt: ____________________
- Erzeugungsweg: Kamera / generiert
- Geplante Variationen: ____________________
- Wodurch unterscheiden sich die Testbilder von den Trainingsbildern? ____________________

Antwort:

## Aufgabe 2 – Ordnerstruktur und Bilder erzeugen

Legen Sie eine klare Struktur an. Erstellen Sie den Split erst nach der Bildproduktion, oder nehmen Sie die drei Splits als getrennte Aufnahmesitzungen auf.

```text
mein_detektionsprojekt/
├── images/
│   ├── train/
│   ├── valid/
│   └── test/
└── annotations/
    ├── train_annotations.json
    ├── valid_annotations.json
    └── test_annotations.json
```

Alle Bilder sollen RGB-PNGs oder -JPEGs sein. Vermeiden Sie sehr kleine Objekte (Faustregel: mindestens etwa 40 × 40 Pixel) und Bilder, bei denen das Objekt abgeschnitten ist – außer Sie möchten genau diesen schwierigen Fall ausdrücklich untersuchen.

**Auszufüllen:**

| Split | Bilder | Aufnahme-/Erzeugungsbedingungen |
| --- | ---: | --- |
| Training |  |  |
| Validierung |  |  |
| Test |  |  |

Wie stellen Sie sicher, dass keine nahezu identischen Bilder in zwei Splits landen?

Antwort:

## Aufgabe 3 – Bounding Boxes annotieren

Zeichnen Sie für jedes Bild eine möglichst enge Box um das Objekt. Sie können ein Annotationstool verwenden oder die Koordinaten bei programmgenerierten Bildern direkt speichern. Verwenden Sie im gesamten Projekt das Format `xyxy`: `[x_min, y_min, x_max, y_max]`; der Ursprung `(0, 0)` liegt oben links.

Speichern Sie je Split eine JSON-Datei. Beispiel für ein Bild mit 640 × 480 Pixeln:

```json
[
  {
    "image": "images/train/tasse_001.jpg",
    "box_xyxy": [125, 80, 450, 420],
    "label": "tasse"
  }
]
```

Für diese Übung muss jede Box gültig sein: `0 ≤ x_min < x_max ≤ Bildbreite` und `0 ≤ y_min < y_max ≤ Bildhöhe`.

## Aufgabe 4 – Labels technisch und visuell prüfen

Laden Sie Ihre JSON-Dateien. Prüfen Sie programmatisch alle Koordinaten auf die Bedingungen aus Aufgabe 3. Visualisieren Sie danach mindestens neun zufällig gewählte Bilder mit eingezeichneter Bounding Box – je drei aus Training, Validierung und Test.

Kontrollieren Sie besonders: falsche Bildpfade, vertauschte Koordinaten, Boxen außerhalb des Bildes, zu große Ränder und falsche Objekte. Beheben Sie die Fehler vor dem Training.

**Auszufüllen:**

- Geprüfte Labels: ____________________
- Gefundene und korrigierte Fehler: ____________________
- Welche Bilder bzw. Boxen sind besonders schwierig? ____________________

Antwort:

## Aufgabe 5 – Dataset und DataLoader bauen

Übernehmen Sie die Idee von `PennFudanSinglePersonDataset` aus dem Tag-9-Notebook und passen Sie sie an Ihre Ordner und JSON-Dateien an. Geben Sie pro Sample ein Bildtensor mit Shape `(3, H, W)` und eine normalisierte Box im Bereich `[0, 1]` zurück.

Verwenden Sie für das Training mindestens einen horizontalen Flip und – falls er zu Ihrem Objekt passt – vorsichtige Helligkeits- und Kontraständerungen. Bei jedem geometrischen Augmentierungsschritt muss die Bounding Box korrekt mittransformiert werden. Test- und Validierungsbilder dürfen nicht augmentiert werden.

**Auszufüllen:**

| Objekt | Shape / Wertebereich |
| --- | --- |
| Bildtensor |  |
| Bounding Box vor Normalisierung |  |
| Bounding Box nach Normalisierung |  |

Warum muss sich die Box bei einem horizontalen Flip ändern?

Antwort:

## Aufgabe 6 – Baseline und Modell trainieren

Starten Sie mit dem `TransferAnchorDetector` aus `01_tag_9_objekterkennung_einfache_anker_nms.ipynb`. Übernehmen Sie die Modell-, Loss- und NMS-Funktionen; passen Sie nur Dataset-Pfade und gegebenenfalls Bildgröße an.

Trainieren Sie zunächst nur den Detektionskopf auf dem vortrainierten ResNet-18-Backbone. Danach dürfen Sie den letzten ResNet-Block mit kleinerer Lernrate feinjustieren. Speichern Sie den Checkpoint mit der besten **Validierungsleistung**, nicht den letzten Trainingsstand.

Falls Sie mehrere Objekte oder Klassen pro Bild umsetzen möchten, ist das ein Bonus: Dann verwenden Sie beispielsweise einen vortrainierten `FasterRCNN`-Detektor und ein Dataset, das Boxen und Klassenlabels pro Bild zurückgibt.

## Aufgabe 7 – Validierung mit IoU

Implementieren Sie die Intersection over Union (IoU) zwischen einer vorhergesagten und der echten Box. Werten Sie die Validierung nach NMS aus und berichten Sie mindestens:

- mittlere IoU aller Validierungsbilder,
- Anteil der Bilder mit IoU ≥ 0,5,
- Anteil der Bilder mit einer Vorhersage oberhalb Ihrer Score-Schwelle,
- eine Galerie aus mindestens sechs Validierungsbildern mit Label (grün), Vorhersage (blau) und Score.

Optimieren Sie Lernrate, Epochenzahl, Augmentation, Score-Schwelle oder Datenvielfalt ausschließlich mit Training und Validierung.

**Auszufüllen:**

| Modellvariante | Augmentation / Hyperparameter | mittlere IoU valid | Trefferquote bei IoU ≥ 0,5 | Beobachtung |
| --- | --- | ---: | ---: | --- |
| Baseline |  |  |  |  |
| Variante 1 |  |  |  |  |
| Variante 2 |  |  |  |  |

Welche Variante wählen Sie – und warum?

Antwort:

## Aufgabe 8 – Finale Prüfung auf ungesehenen Testbildern

Laden Sie genau den anhand der Validierung ausgewählten Checkpoint. Bestimmen Sie die Kennzahlen nun **einmalig** auf dem Testsplit. Zeigen Sie zusätzlich alle Testbilder oder mindestens 15 zufällig ausgewählte Testbilder mit echten und vorhergesagten Boxen.

Als gutes Kursziel gelten eine mittlere Test-IoU von mindestens **0,50** und eine Trefferquote von mindestens **80 %** bei IoU ≥ 0,5. Falls Sie das Ziel nicht erreichen, dokumentieren Sie ehrlich die Fehler und verbessern Sie nur über einen neuen Trainings-/Validierungslauf – nicht durch Anpassung an die Testbilder.

**Auszufüllen – finale Abgabe:**

| Kennzahl | Validierung | Test (ungesehen) |
| --- | ---: | ---: |
| mittlere IoU |  |  |
| Trefferquote bei IoU ≥ 0,5 |  |  |
| Bilder mit Vorhersage über Score-Schwelle |  |  |

- Finales Modell / wichtigste Hyperparameter: ____________________
- Erreicht das Modell das Kursziel? ____________________
- Drei typische Fehlerfälle: ____________________
- Nächste sinnvolle Verbesserung: ____________________

Antwort: